# MMAF-CD — External-data importer (run ONCE on Kaggle)

Four external test sets are not hosted on Kaggle. This notebook downloads them, converts 3-D scans to 2-D PNG
slices with masks and a `labels.csv`, and writes everything to `/kaggle/working/mmaf-external-extra/`.

**Settings:** Internet **ON**, accelerator **None** (CPU is enough), persistence *Files only*.

**After it finishes:** *Save Version → Save & Run All*, then open the notebook's **Output** tab →
**New Dataset** → name it `mmaf-external-extra` (private). Add that dataset as an input to the v21 notebook.
Optionally, put your Kaggle API key in a secret called `KAGGLE_KEY_JSON` and the last cell creates the dataset for you.

| Folder | Source | Content |
|---|---|---|
| `gastric_gashissdb/` | figshare 15066147 (GasHisSDB, 4.4 GB .rar) | 160×160 Normal/Abnormal H&E tiles |
| `bladder_fedbca/` | Zenodo 10409145 (FedBCa, 4 centres) | T2WI max-tumour slices + masks + MIBC/NMIBC labels |
| `pancreas_msd07/` | Medical Segmentation Decathlon Task07 | CT tumour slices + masks |
| `oral_ndb_ufes/` | Mendeley bbmmm4wgr8 v4 (NDB-UFES) | OSCC / leukoplakia patches (manual upload if the API fails) |


In [ ]:
import subprocess, sys, os, re, json, shutil, tarfile, zipfile, glob
from pathlib import Path
def sh(c): print("$", c); return subprocess.run(c, shell=True).returncode
for pk in ["nibabel", "requests", "pandas", "openpyxl"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pk])
import numpy as np, pandas as pd, requests, nibabel as nib
from PIL import Image
OUT = Path("/kaggle/working/mmaf-external-extra"); OUT.mkdir(parents=True, exist_ok=True)
TMP = Path("/tmp/mmaf_dl"); TMP.mkdir(parents=True, exist_ok=True)

def download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0: return dest
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status(); tot = int(r.headers.get("content-length", 0)); got = 0
        with open(dest, "wb") as f:
            for ch in r.iter_content(8 << 20):
                f.write(ch); got += len(ch)
                if tot and got % (512 << 20) < (8 << 20): print(f"   {got/1e9:.2f}/{tot/1e9:.2f} GB")
    return dest

def unrar(archive, out):
    Path(out).mkdir(parents=True, exist_ok=True)
    for cmd in [f"unrar x -o+ -inul '{archive}' '{out}/'", f"7z x -y '{archive}' -o'{out}'", f"bsdtar -xf '{archive}' -C '{out}'"]:
        if shutil.which(cmd.split()[0]) and sh(cmd) == 0: return True
    sh("apt-get -qq update && apt-get -qq install -y unrar p7zip-full >/dev/null 2>&1")
    return sh(f"unrar x -o+ -inul '{archive}' '{out}/'") == 0 or sh(f"7z x -y '{archive}' -o'{out}'") == 0

def slices(img_nii, lab_nii, out_dir, tag, window=None, tumour_label=None, k=3):
    img = nib.load(str(img_nii)).get_fdata(); lab = nib.load(str(lab_nii)).get_fdata()
    tgt = (lab == tumour_label) if tumour_label is not None else (lab > 0)
    area = tgt.sum((0, 1)); rows = []
    for z in [z for z in np.argsort(-area)[:k] if area[z] > 0]:
        s = img[:, :, z].astype(np.float32)
        if window: s = np.clip((s - window[0]) / (window[1] - window[0]), 0, 1)
        else: lo, hi = np.percentile(s, [1, 99]); s = np.clip((s - lo) / (hi - lo + 1e-6), 0, 1)
        im = np.rot90((s * 255).astype(np.uint8)); m = np.rot90(tgt[:, :, z].astype(np.uint8) * 255)
        ip, mp = out_dir / f"{tag}_z{z}.png", out_dir / f"{tag}_z{z}_mask.png"
        Image.fromarray(im).convert("RGB").save(ip); Image.fromarray(m).save(mp)
        rows.append((ip.relative_to(out_dir).as_posix(), mp.relative_to(out_dir).as_posix()))
    return rows
print("disk:", shutil.disk_usage("/tmp").free / 1e9, "GB free in /tmp;", shutil.disk_usage("/kaggle/working").free / 1e9, "GB in working")


In [ ]:
# ── GasHisSDB (figshare 15066147) — keep only the 160×160 tiles ──
dst = OUT / "gastric_gashissdb"
if not dst.exists():
    rar = download("https://ndownloader.figshare.com/files/28969725", TMP / "GasHisSDB.rar")
    ok = unrar(rar, TMP / "gashis")
    print("extracted:", ok)
    src = next((Path(p) for p in glob.glob(str(TMP / "gashis/**/160"), recursive=True) if Path(p).is_dir()), None)
    if src is None: print("⚠️ 160 folder not found; layout:"); sh(f"find {TMP/'gashis'} -maxdepth 3 -type d | head -30")
    else:
        shutil.copytree(src, dst); rar.unlink(missing_ok=True); shutil.rmtree(TMP / "gashis", ignore_errors=True)
print({d.name: len(list(d.glob('*'))) for d in dst.iterdir() if d.is_dir()} if dst.exists() else "missing")


In [ ]:
# ── FedBCa (Zenodo 10409145) → max-tumour T2WI slices + masks + labels.csv ──
dst = OUT / "bladder_fedbca"; dst.mkdir(exist_ok=True)
raw = TMP / "fedbca"; raw.mkdir(exist_ok=True)
meta = requests.get("https://zenodo.org/api/records/10409145", timeout=60).json()
for f in meta.get("files", []):
    key = f["key"]; print("  file:", key, f"{f['size']/1e9:.2f} GB")
    p = download(f["links"]["self"], raw / key)
    if key.endswith(".zip"): zipfile.ZipFile(p).extractall(raw)
    elif key.endswith((".tar", ".tar.gz", ".tgz")): tarfile.open(p).extractall(raw)
    elif key.endswith(".rar"): unrar(p, raw)
# label tables
lab = {}
for t in list(raw.rglob("*.xlsx")) + list(raw.rglob("*.csv")):
    try: df = pd.read_excel(t) if t.suffix == ".xlsx" else pd.read_csv(t)
    except Exception: continue
    idc = [c for c in df.columns if re.search(r"id|name|case|patient", str(c), re.I)]
    lc = [c for c in df.columns if re.search(r"mibc|invas|label|stage|muscle|patholog", str(c), re.I)]
    print(f"  table {t.name}: columns={list(df.columns)[:10]}")
    if not idc or not lc: continue
    for _, r in df.iterrows():
        v = str(r[lc[0]]).strip().lower()
        y = "muscle_invasive" if (v in {"1", "mibc", "yes"} or v.startswith("mibc") or v.startswith("t2") or v.startswith("t3") or v.startswith("t4")) else \
            ("non_muscle_invasive" if (v in {"0", "nmibc", "no"} or v.startswith("nmibc") or v.startswith("ta") or v.startswith("t1") or v.startswith("tis")) else None)
        if y: lab[str(r[idc[0]]).strip()] = y
print("  labelled cases:", len(lab), pd.Series(list(lab.values())).value_counts().to_dict() if lab else "")
niis = list(raw.rglob("*.nii*")); rows = []
for p in [q for q in niis if not re.search(r"mask|label|seg|roi", q.name, re.I)]:
    cid = re.sub(r"\.nii(\.gz)?$", "", p.name)
    key = next((k for k in lab if k and (k == cid or k in cid or cid in k)), None)
    m = next((q for q in niis if q != p and cid in q.name and re.search(r"mask|label|seg|roi", q.name, re.I)), None)
    if key is None or m is None: continue
    for ip, mp in slices(p, m, dst, cid):
        rows.append({"image": ip, "label": lab[key], "mask": mp, "case": cid})
pd.DataFrame(rows).to_csv(dst / "labels.csv", index=False)
print(f"  ✅ FedBCa: {len(rows)} slices from {len({r['case'] for r in rows})} cases"
      if rows else "  ⚠️ no slices written — check the table/column printout above and adapt the label regex")


In [ ]:
# ── MSD Task07 Pancreas (AWS, 11 GB tar) → tumour slices + masks (streamed, low disk) ──
N_CASES = 120
dst = OUT / "pancreas_msd07"; dst.mkdir(exist_ok=True)
if not (dst / "labels.csv").exists():
    url = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task07_Pancreas.tar"
    keep = TMP / "msd"; keep.mkdir(exist_ok=True); n_img = 0
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with tarfile.open(fileobj=r.raw, mode="r|*") as tf:
            for m in tf:
                mm = re.search(r"(imagesTr|labelsTr)/pancreas_(\d+)\.nii\.gz$", m.name)
                if not mm or m.name.split("/")[-1].startswith("._"): continue
                if int(mm.group(2)) > 3 * N_CASES: continue
                tf.extract(m, keep); n_img += 1
    rows = []
    for ip in sorted((keep).rglob("imagesTr/pancreas_*.nii.gz"))[:N_CASES]:
        lp = ip.parent.parent / "labelsTr" / ip.name
        if not lp.exists(): continue
        for a, b in slices(ip, lp, dst, ip.name.split(".")[0], window=(-100, 240), tumour_label=2, k=2):
            rows.append({"image": a, "label": "tumor", "mask": b, "case": ip.name})
    pd.DataFrame(rows).to_csv(dst / "labels.csv", index=False); shutil.rmtree(keep, ignore_errors=True)
    print(f"  ✅ MSD07: {len(rows)} tumour slices")


In [ ]:
# ── NDB-UFES (Mendeley bbmmm4wgr8 v4) — oral OSCC vs leukoplakia ──
dst = OUT / "oral_ndb_ufes"
if not dst.exists():
    try:
        files = requests.get("https://data.mendeley.com/public-api/datasets/bbmmm4wgr8/files",
                             params={"folder_id": "root", "version": 4}, timeout=60).json()
        dst.mkdir()
        for f in files:
            url = f.get("content_details", {}).get("download_url")
            if not url: continue
            p = download(url, TMP / f["filename"]); print("  got", f["filename"])
            if f["filename"].endswith(".zip"): zipfile.ZipFile(p).extractall(dst)
            elif f["filename"].endswith(".rar"): unrar(p, dst)
            else: shutil.copy(p, dst / f["filename"])
        print("  ✅ NDB-UFES:", sum(1 for _ in dst.rglob("*.png")) + sum(1 for _ in dst.rglob("*.jpg")), "images")
    except Exception as e:
        shutil.rmtree(dst, ignore_errors=True)
        print(f"  ⚠️ Mendeley API failed ({e}).\n     Download https://data.mendeley.com/datasets/bbmmm4wgr8/4 (Download All) in your browser,\n"
              f"     then add the zip to the Kaggle dataset 'mmaf-external-extra' in a folder named oral_ndb_ufes/.")


In [ ]:
# ── summary + (optional) create the Kaggle dataset via API ──
for d in sorted(OUT.iterdir()):
    n = sum(1 for p in d.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".bmp"})
    print(f"  {d.name:<22} {n:>7} images   {sum(p.stat().st_size for p in d.rglob('*') if p.is_file())/1e9:.2f} GB")
(OUT / "README.md").write_text("External test sets for MMAF-CD v21 (GasHisSDB 160px, FedBCa slices, MSD Task07 slices, NDB-UFES). "
                               "Original licences apply: GasHisSDB CC BY 4.0 (figshare), FedBCa CC BY 4.0 (Zenodo), MSD CC BY-SA 4.0, NDB-UFES CC BY 4.0.")
try:
    from kaggle_secrets import UserSecretsClient
    cred = json.loads(UserSecretsClient().get_secret("KAGGLE_KEY_JSON"))
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    json.dump(cred, open(os.path.expanduser("~/.kaggle/kaggle.json"), "w")); os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    json.dump({"title": "mmaf-external-extra", "id": f"{cred['username']}/mmaf-external-extra",
               "licenses": [{"name": "CC-BY-SA-4.0"}]}, open(OUT / "dataset-metadata.json", "w"))
    sh(f"kaggle datasets create -p {OUT} --dir-mode zip -q || kaggle datasets version -p {OUT} -m update --dir-mode zip -q")
except Exception as e:
    print("No KAGGLE_KEY_JSON secret — create the dataset from this notebook's Output tab instead.", e)
